<a href="https://colab.research.google.com/github/trsantanarural/predicao-falhas-tarefas/blob/main/notebooks/04_agrupamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04. Agrupamento: K-Means e DBSCAN

**Entrada:** `saida/base_modelagem.parquet` (notebook 02)
**Saídas:** `saida/base_conjuntos.parquet` (conjunto de cada tarefa, usado também no notebook 05), `saida/agrupamento_*.csv` e figuras em `figuras/`

Este notebook:
1. atribui cada job por inteiro a um conjunto (treino, validação ou teste), pelo dia da sua primeira tarefa;
2. prepara as métricas de consumo (imputação, `log(1 + x)` e padronização, ajustadas só no treino);
3. ajusta o K-Means e o DBSCAN no treino (dias 1 a 19), sem o rótulo;
4. associa as tarefas de validação e de teste aos grupos e avalia o escore de atipicidade (distância ao centroide ou ao ponto central mais próximo) por ROC-AUC e PR-AUC, além da proporção de falhas por grupo.

Os critérios de escolha de `minPts` e de ε estão fixados antes de olhar o resultado: `minPts` = 2 × número de atributos e ε no cotovelo da curva de distâncias ao k-ésimo vizinho (ponto de maior afastamento em relação à reta que liga as extremidades da curva).

## 1. Configuração do ambiente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, roc_auc_score, average_precision_score

# Pasta de trabalho no Google Drive (base do notebook 02, saídas e figuras)
BASE = '/content/drive/MyDrive/artigo_predicao'
FIG = f'{BASE}/figuras'
SAIDA = f'{BASE}/saida'
os.makedirs(FIG, exist_ok=True)

SEED = 42
KEY = ['job_id', 'task_index']

# Métricas de consumo usadas no agrupamento (sem tempo_obs e n_janelas, que carregam a duração)
CONSUMO = ['cpu_rate', 'canonical_mem', 'assigned_mem', 'unmapped_cache', 'total_cache',
           'disk_space', 'cpi', 'mai', 'sampled_cpu', 'max_mem', 'max_cpu']
NOMES = {'cpu_rate': 'CPU média', 'canonical_mem': 'Memória canônica', 'assigned_mem': 'Memória atribuída',
         'unmapped_cache': 'Cache não mapeado', 'total_cache': 'Cache total', 'disk_space': 'Espaço em disco',
         'cpi': 'Ciclos por instrução', 'mai': 'Acessos à memória por instrução',
         'sampled_cpu': 'Amostra de CPU', 'max_mem': 'Memória máxima', 'max_cpu': 'CPU máxima'}

AZUL, LARANJA = '#2a78d6', '#eb6834'
TXT, TXT2, GRADE = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({
    'font.size': 10, 'axes.edgecolor': TXT2, 'axes.labelcolor': TXT, 'xtick.color': TXT2,
    'ytick.color': TXT2, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRADE, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.bbox': 'tight'})

## 2. Divisão por job
Cada job vai por inteiro para o conjunto do dia da sua primeira tarefa (menor `dia` entre as tarefas do job na base): dias 1 a 19 no treino, 20 a 23 na validação e 24 a 29 no teste. Isso elimina a sobreposição de jobs entre os conjuntos (na divisão por dia da tarefa, usada na análise exploratória, 9 jobs ficavam em mais de um conjunto).

In [ ]:
base = pd.read_parquet(f'{SAIDA}/base_modelagem.parquet')

dia_job = base.groupby('job_id').dia.min().rename('dia_job')
base = base.join(dia_job, on='job_id')
base['conjunto'] = np.select([base.dia_job >= 24, base.dia_job >= 20], ['teste', 'validação'], 'treino')

# Comparação com a divisão por dia da tarefa
por_tarefa = np.select([base.dia >= 24, base.dia >= 20], ['teste', 'validação'], 'treino')
print('tarefas que mudam de conjunto em relação à divisão por dia da tarefa:', int((por_tarefa != base.conjunto).sum()))

resumo = base.groupby('conjunto').falha.agg(tarefas='size', falhas='sum', taxa='mean')
resumo['jobs'] = base.groupby('conjunto').job_id.nunique()
print(resumo.reindex(['treino', 'validação', 'teste']).round(4).to_string())

base[KEY + ['dia', 'dia_job', 'conjunto', 'falha']].to_parquet(f'{SAIDA}/base_conjuntos.parquet', index=False)

## 3. Preparação das métricas de consumo
Os valores ausentes de CPI e MAI são imputados pela mediana do treino; em seguida aplica-se `log(1 + x)` e a padronização com média e desvio padrão do treino. O peso de cada tarefa é o inverso do número de tarefas do seu job na base.

In [ ]:
tr = base[base.conjunto == 'treino'].copy()

medianas = tr[CONSUMO].median()                      # mediana do treino (imputação)
def prepara(df):
    x = df[CONSUMO].fillna(medianas)
    return np.log1p(x.to_numpy(dtype='float64'))

escala = StandardScaler().fit(prepara(tr))
def padroniza(df):
    return escala.transform(prepara(df))

n_job = base.groupby('job_id').size()
base['peso'] = 1.0 / base.job_id.map(n_job)
tr = base[base.conjunto == 'treino'].copy()
X_tr = padroniza(tr)
print('treino:', X_tr.shape, '| jobs:', tr.job_id.nunique())

## 4. Amostra do treino com limite de tarefas por job
O DBSCAN tem custo quadrático nas vizinhanças e, sem limite, os poucos jobs com milhares de tarefas quase idênticas formariam os grupos densos. Cada job contribui com no máximo `LIMITE` tarefas (amostra aleatória, semente fixa). O limite é o primeiro de 50, 20, 10 e 5 que deixa a amostra com até 100.000 pontos.

In [ ]:
MAX_PONTOS = 100_000
embaralhada = tr.sample(frac=1.0, random_state=SEED)
for LIMITE in (50, 20, 10, 5):
    amostra = embaralhada[embaralhada.groupby('job_id').cumcount() < LIMITE]
    print(f'limite {LIMITE:>2} tarefas por job -> {len(amostra):,} pontos'.replace(',', '.'))
    if len(amostra) <= MAX_PONTOS:
        break
print('LIMITE adotado:', LIMITE, '| pontos:', len(amostra), '| taxa de falha na amostra:', round(amostra.falha.mean(), 4))
X_am = padroniza(amostra)

## 5. K-Means
Ajustado nas tarefas de treino com peso por job, para k de 2 a 10. A silhueta média é calculada em uma subamostra de até 20.000 pontos da amostra limitada por job (o custo é quadrático). O k adotado é o de maior silhueta; o gráfico do cotovelo (inércia ponderada) acompanha a escolha. Se, olhando os dois gráficos, outro k for preferível, defina `K_ESCOLHIDO` manualmente e justifique no artigo.

In [ ]:
rng = np.random.RandomState(SEED)
idx_sil = rng.choice(len(X_am), size=min(20_000, len(X_am)), replace=False)

ks = list(range(2, 11))
inercia, silueta = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=3, random_state=SEED).fit(X_tr, sample_weight=tr.peso.to_numpy())
    inercia.append(km.inertia_)
    silueta.append(silhouette_score(X_am[idx_sil], km.predict(X_am[idx_sil])))
    print(f'k = {k:>2} | inércia ponderada = {km.inertia_:,.1f} | silhueta = {silueta[-1]:.4f}'.replace(',', ' '))

fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
axs[0].plot(ks, inercia, marker='o', color=AZUL); axs[0].set_xlabel('k'); axs[0].set_ylabel('Inércia ponderada')
axs[1].plot(ks, silueta, marker='o', color=LARANJA); axs[1].set_xlabel('k'); axs[1].set_ylabel('Silhueta média')
for ax in axs: ax.set_xticks(ks)
fig.tight_layout()
fig.savefig(f'{FIG}/kmeans_cotovelo_silhueta.png', dpi=200)
plt.show()

In [ ]:
K_ESCOLHIDO = None          # defina um inteiro para substituir a escolha automática
K = K_ESCOLHIDO or ks[int(np.argmax(silueta))]
print('k adotado:', K)

km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X_tr, sample_weight=tr.peso.to_numpy())
tr['grupo'] = km.labels_

# Descrição dos grupos no treino: tamanho, taxa de falha e centroide (escala original aproximada)
cent = pd.DataFrame(escala.inverse_transform(km.cluster_centers_), columns=CONSUMO)
cent = np.expm1(cent)
cent.columns = [NOMES[c] for c in CONSUMO]
desc = tr.groupby('grupo').falha.agg(tarefas='size', falhas='sum', taxa='mean').round(4)
desc['jobs'] = tr.groupby('grupo').job_id.nunique()
print(desc.to_string())
print()
print(cent.round(5).T.to_string())

## 6. DBSCAN
`minPts` = 2 × número de atributos. O ε é lido na curva das distâncias ao `minPts`-ésimo vizinho, ordenadas, no ponto de maior afastamento em relação à reta que liga as extremidades da curva.

In [ ]:
MINPTS = 2 * len(CONSUMO)
nn = NearestNeighbors(n_neighbors=MINPTS).fit(X_am)
kdist = np.sort(nn.kneighbors(X_am)[0][:, -1])

# cotovelo: ponto da curva mais distante da reta entre as extremidades
xn = np.arange(len(kdist)) / (len(kdist) - 1)
yn = (kdist - kdist[0]) / (kdist[-1] - kdist[0])
i_joelho = int(np.argmax(xn - yn))
EPS_AUTO = float(kdist[i_joelho])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(kdist, color=AZUL, lw=1.4)
ax.axhline(EPS_AUTO, ls='--', lw=1, color=LARANJA)
ax.text(len(kdist) * 0.02, EPS_AUTO, f' ε = {EPS_AUTO:.3f}'.replace('.', ','), va='bottom', color=LARANJA)
ax.set_xlabel('Pontos da amostra, em ordem crescente'); ax.set_ylabel(f'Distância ao {MINPTS}º vizinho')
fig.savefig(f'{FIG}/dbscan_kdistance.png', dpi=200)
plt.show()
print('minPts:', MINPTS, '| ε sugerido pelo cotovelo:', round(EPS_AUTO, 4))

In [ ]:
EPS_ESCOLHIDO = None        # defina um número para substituir o valor do cotovelo
EPS = EPS_ESCOLHIDO or EPS_AUTO
print('ε adotado:', round(EPS, 4))

# Verificação de memória: o DBSCAN guarda a vizinhança de cada ponto (n x tamanho médio da vizinhança).
sond = np.random.RandomState(SEED).choice(len(X_am), size=min(500, len(X_am)), replace=False)
viz = NearestNeighbors(radius=EPS).fit(X_am).radius_neighbors(X_am[sond], return_distance=False)
media_viz = float(np.mean([len(v) for v in viz]))
mem_gb = len(X_am) * media_viz * 8 / 1e9
print(f'vizinhos médios dentro de ε: {media_viz:.0f} | memória estimada das vizinhanças: {mem_gb:.1f} GB')
if mem_gb > 4:
    raise RuntimeError('ε grande demais para a amostra: o DBSCAN ficaria sem memória. '
                       'Reduza MAX_PONTOS (seção 4) ou defina EPS_ESCOLHIDO e justifique no artigo.')

db = DBSCAN(eps=EPS, min_samples=MINPTS, n_jobs=-1).fit(X_am)
amostra = amostra.assign(grupo_db=db.labels_)
n_grupos = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print('grupos:', n_grupos, '| pontos centrais:', len(db.core_sample_indices_),
      '| ruído (%):', round((db.labels_ == -1).mean() * 100, 1))

d = amostra.groupby('grupo_db').falha.agg(tarefas='size', falhas='sum', taxa='mean').round(4)
d['jobs'] = amostra.groupby('grupo_db').job_id.nunique()
print(d.rename(index={-1: 'ruído'}).to_string())

## 7. Associação das tarefas de validação e de teste e avaliação
- **K-Means:** cada tarefa vai ao centroide mais próximo; o escore de atipicidade é a distância a ele.
- **DBSCAN:** o algoritmo não atribui observações novas; cada tarefa é associada ao ponto central mais próximo, se ele estiver a menos de ε, e rotulada como ruído caso contrário. O escore de atipicidade é a distância ao ponto central mais próximo.

ROC-AUC e PR-AUC avaliam se o escore ordena as tarefas com falha acima das concluídas, sem uso do rótulo no ajuste. A PR-AUC de um escore sem informação é igual à taxa de falha do conjunto. As métricas aparecem também sem o job com mais falhas do conjunto, que no teste concentra a maior parte delas.

In [ ]:
# Pontos centrais do DBSCAN e seus rótulos
centrais = db.components_
rot_centrais = db.labels_[db.core_sample_indices_]
nn_c = NearestNeighbors(n_neighbors=1).fit(centrais)

def aplica(df):
    X = padroniza(df)
    d_km = km.transform(X)
    g_km, s_km = d_km.argmin(axis=1), d_km.min(axis=1)
    dist, pos = nn_c.kneighbors(X)
    dist, pos = dist[:, 0], pos[:, 0]
    g_db = np.where(dist < EPS, rot_centrais[pos], -1)
    return g_km, s_km, g_db, dist

def metricas(y, s):
    return roc_auc_score(y, s), average_precision_score(y, s)

linhas, grupos_tab = [], []
for conj in ('validação', 'teste'):
    df = base[base.conjunto == conj].copy()
    g_km, s_km, g_db, s_db = aplica(df)
    df['grupo_km'], df['esc_km'], df['grupo_db'], df['esc_db'] = g_km, s_km, g_db, s_db

    # job com mais falhas do conjunto
    top = df[df.falha == 1].groupby('job_id').size().idxmax()
    for rotulo, sub in (('todos os jobs', df), (f'sem o job {top}', df[df.job_id != top])):
        for nome, col in (('K-Means', 'esc_km'), ('DBSCAN', 'esc_db')):
            roc, pr = metricas(sub.falha, sub[col])
            linhas.append({'conjunto': conj, 'recorte': rotulo, 'método': nome, 'tarefas': len(sub),
                           'taxa de falha': round(sub.falha.mean(), 4),
                           'ROC-AUC': round(roc, 4), 'PR-AUC': round(pr, 4)})
        ruido = sub[sub.grupo_db == -1]
        linhas.append({'conjunto': conj, 'recorte': rotulo, 'método': 'DBSCAN: pontos de ruído',
                       'tarefas': len(ruido), 'taxa de falha': round(ruido.falha.mean(), 4) if len(ruido) else np.nan,
                       'ROC-AUC': np.nan, 'PR-AUC': np.nan})

    for nome, col in (('K-Means', 'grupo_km'), ('DBSCAN', 'grupo_db')):
        t = df.groupby(col).falha.agg(tarefas='size', falhas='sum', taxa='mean').round(4).reset_index()
        t.insert(0, 'método', nome); t.insert(0, 'conjunto', conj); t = t.rename(columns={col: 'grupo'})
        grupos_tab.append(t)

res = pd.DataFrame(linhas)
print(res.to_string(index=False))

In [ ]:
grupos_tab = pd.concat(grupos_tab, ignore_index=True)
grupos_tab['grupo'] = grupos_tab.grupo.astype(str).replace({'-1': 'ruído'})
print(grupos_tab.to_string(index=False))

## 8. Gravação dos resultados

In [ ]:
res.to_csv(f'{SAIDA}/agrupamento_metricas.csv', index=False)
grupos_tab.to_csv(f'{SAIDA}/agrupamento_grupos.csv', index=False)
pd.DataFrame({'parametro': ['k', 'limite_por_job', 'pontos_amostra', 'minPts', 'eps'],
              'valor': [K, LIMITE, len(amostra), MINPTS, EPS]}).to_csv(f'{SAIDA}/agrupamento_parametros.csv', index=False)
print('gravado em', SAIDA)